# Homework-4: Building and evaluating a network IDS based on Network Traffic characterstics

**Group Size**: Up to two (2)

**Group Members (enter your names below)**:
 - Todd Sekmistrz

**Notes:**
- You are free to re-use code from the last Network-IDS demo (uploaded to canvas).
- Please submit a modified version of this notebook with all the code and answers. Make sure that your code works well before you submit the notebook.
- To avoid dependency issues, it is better to work on this homework using Google colab.
- For all models you train, don't do any parameter tuning. Just use the default because your answers will be evaluated against the default instances of the models.

**Homework #4: Training Classification Models For An IDS**

**Course: CIS 540 - Foundations of Information Security**

**Name: Todd Sekmistrz**


In [1]:
# Imports you likely need
import pandas as pd
import numpy as np
from io import StringIO
import requests

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

import matplotlib.pyplot as plt
import itertools
import warnings

In [2]:
# function to read data (same dataset used in Network-IDS-Demo)

def load_from_gdrive(URL):
  path_to_csv = 'https://drive.google.com/uc?id='+URL.split('/')[-2]
  content = requests.get(path_to_csv).text
  csv_raw = StringIO(content)
  df = pd.read_csv(csv_raw)
  return df


In [3]:
def load_and_merge():
    warnings.filterwarnings('ignore') # for instance: google virus scan warning etc.
    url1 = 'https://drive.google.com/file/d/1IOIUbgt1aq3W2MLYsbBu3TNYXpwdhiVt/view?usp=sharing' # Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
    url2 = 'https://drive.google.com/file/d/1wguITi4enxKLqI0CKdwh5GRSiTcBzPct/view?usp=sharing' # Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
    url3 = 'https://drive.google.com/file/d/1rOVLNLsIPoapO-6XoAZYsf6IRz_sfzkh/view?usp=sharing' # Friday-WorkingHours-Morning.pcap_ISCX.csv
    url4 = 'https://drive.google.com/file/d/1d_FUc6oWlUgRmnv7wtssI1K0OI9cdfpu/view?usp=sharing' # Monday-WorkingHours.pcap_ISCX_1.csv
    url5 = 'https://drive.google.com/file/d/1hnnXJWwYJg95vll463wqPq67DE369V_h/view?usp=sharing' # Monday-WorkingHours.pcap_ISCX_2.csv
    url6 = 'https://drive.google.com/file/d/1IGwVQA0e8BwkeNDrMvQsaWCeIqhlAc6G/view?usp=sharing' # Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
    url7 = 'https://drive.google.com/file/d/1xwt_TIHMPDhDZQEu29DxKAvcqm61ygwo/view?usp=sharing' # Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
    url8 = 'https://drive.google.com/file/d/1sJejwWsYFyJMsDD3bQAQ4u5plhv8DF7r/view?usp=sharing' # Tuesday-WorkingHours.pcap_ISCX_1.csv
    url9 = 'https://drive.google.com/file/d/1kfl0F9LGGXUf-4gl3BL-VndCnggaL9n_/view?usp=sharing' # Tuesday-WorkingHours.pcap_ISCX_2.csv
    url10 = 'https://drive.google.com/file/d/1nYbDWeYHf3VrBv8haYI1XPlniZcrV2zw/view?usp=sharing' # Wednesday-workingHours.pcap_ISCX_1.csv
    url11 = 'https://drive.google.com/file/d/1S7iVzYfA9W_ucWwYt7CtI-wsIaABwf2A/view?usp=sharing' # Wednesday-workingHours.pcap_ISCX_2.csv
    url12 = 'https://drive.google.com/file/d/1eMbeamPC29ys2ayQ2--JENDVNpsvkGv9/view?usp=sharing' # Wednesday-workingHours.pcap_ISCX_3.csv

    urls = [url1, url2,url3,url4,url5,url6,url7,url8,url9,url10,url11,url12]
    dframes =[]
    for url in urls:
        df= load_from_gdrive(url)
        dframes.append(df)
        print ('Done loading from: '+url)

    return pd.concat(dframes)


In [4]:
dataset = load_and_merge()
print (dataset.shape)

Done loading from: https://drive.google.com/file/d/1IOIUbgt1aq3W2MLYsbBu3TNYXpwdhiVt/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1wguITi4enxKLqI0CKdwh5GRSiTcBzPct/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1rOVLNLsIPoapO-6XoAZYsf6IRz_sfzkh/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1d_FUc6oWlUgRmnv7wtssI1K0OI9cdfpu/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1hnnXJWwYJg95vll463wqPq67DE369V_h/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1IGwVQA0e8BwkeNDrMvQsaWCeIqhlAc6G/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1xwt_TIHMPDhDZQEu29DxKAvcqm61ygwo/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1sJejwWsYFyJMsDD3bQAQ4u5plhv8DF7r/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1kfl0F9LGGXUf-4gl3BL-VndCnggaL9n_/view?usp=sharing
Done loading from: https://drive.google.com/file/d/1nYbDWeYHf3VrBv8haYI1XPlniZcrV2

In [5]:
# Drop the 80th feature that accidentally got in the list when pd.concat(dframes) was invoked.
dataset = dataset.drop(dataset.columns[-1],axis=1)
print (dataset.shape)

# Consolidate all attack traces to one class called 'Malicious'
dataset['Class'] = np.where(dataset[' Label']=='BENIGN', 'Benign', 'Malicious')

# Define features to be used by the classifier
features = pd.Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count',
       ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count',
       ' ACK Flag Count', ' URG Flag Count', ' CWE Flag Count',
       ' ECE Flag Count', ' Down/Up Ratio', ' Average Packet Size',
       ' Avg Fwd Segment Size', ' Avg Bwd Segment Size',
       ' Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk',
       ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk',
       'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', ' Subflow Fwd Bytes',
       ' Subflow Bwd Packets', ' Subflow Bwd Bytes', 'Init_Win_bytes_forward',
       ' Init_Win_bytes_backward', ' act_data_pkt_fwd',
       ' min_seg_size_forward', 'Active Mean', ' Active Std', ' Active Max',
       ' Active Min', 'Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min'])

(2830743, 79)


In [6]:
dataset.replace([np.inf, -np.inf], np.nan, inplace=True)
print("Columns with invalid values: ", list(dataset.columns[dataset.isna().any()]))
dataset.dropna(inplace=True)

Columns with invalid values:  [' Destination Port', 'Flow Bytes/s', ' Flow Packets/s']


# Question 1 (20pts.)
Train a model using Logistic Regression, Decision Tree, Random Forest, and Multi Layer Perceptron with **default model parameters (no parameter tuning) and all features**. Based on an evaluation of the four models on a 15% test set, answer the following questions:

 - a) Which model has the highest classification accuracy?
 - b) Which model has the highest false positive rate?
 - c) Which model has the lowest false negative rate?
 - d) If you have to make the decision to deploy one of these models to detect intrusions on a network, which one would you pick and why?

**Note**: You need to write the relevant code and show execution outputs to justify your your answers.

In [7]:
dataset.shape

(1730329, 80)

In [8]:
#Split dataset into train and test set. Note Test Size = 15% of dataset.
train, test = train_test_split(dataset, test_size=0.15)


In [9]:

#Dictionary to Store Results
results = {}

#Random Forest
#print("--------------------------Random Forest------------------------------")
if 'train' not in locals() or 'test' not in locals():
  print("Error: 'train' and 'test' DataFrames are not defined. Please run the preceding cells to define them.")
else:
  model_rf = RandomForestClassifier(random_state=0) #Specify the model used
  model_rf.fit(train[features], train['Class'])  #Train the model
  predictions_rf = model_rf.predict(test[features]) #Make Predictions

  #Calculate Accuracy and F1 Metrics
  accuracy_rf = accuracy_score(test['Class'], predictions_rf) #Compare prediction to known label
  #print(f"Accuracy: {accuracy_rf}")
  f1_rf = f1_score(test['Class'], predictions_rf, pos_label = 'Malicious') #Compute F1 Score
  #print(f"F1 Score: {f1_rf}")

  #Make Confusion Matrix
  con_mtrx_rf = pd.crosstab(test['Class'], predictions_rf, rownames = ['Actual'], colnames = ['Predicted'])
  #display(con_mtrx_rf) #Display Confusion Matrix

  #True Positive, False Positive, False Negative, and True Negative
  # .loc[row (Actuals), column (Predictions)]
  true_pos_rf = con_mtrx_rf.loc['Malicious', 'Malicious']
  false_pos_rf = con_mtrx_rf.loc['Benign', 'Malicious']
  false_neg_rf = con_mtrx_rf.loc['Malicious', 'Benign']
  true_neg_rf = con_mtrx_rf.loc['Benign', 'Benign']

  #False Positive Rate
  false_pos_rate_rf = false_pos_rf / (false_pos_rf + true_neg_rf)
  #print(f"False Positive Rate: {false_pos_rate_rf}")

  #False Negative Rate
  false_neg_rate_rf = false_neg_rf / (false_neg_rf + true_pos_rf)
  #print(f"False Negative Rate: {false_neg_rate_rf}")

  #Store Results for Random Forest
  results['random_forest'] = {"Accuracy": accuracy_rf, "F1 Score": f1_rf, "False Positive Rate": false_pos_rate_rf, "False Negative Rate": false_neg_rate_rf, "Confusion Matrix": con_mtrx_rf}

  print("Random Forest - Done.")

#print("\n--------------------------Decision Tree------------------------------")
if 'train' not in locals() or 'test' not in locals():
  print("Error: 'train' and 'test' DataFrames are not defined. Please run the preceding cells to define them.")
else:
  #Note: Decision Tree Classifier does not take n_jobs argument
  model_dt = DecisionTreeClassifier(random_state = 0) #Specify the model used
  model_dt.fit(train[features], train['Class']) #Train the model
  predictions_dt = model_dt.predict(test[features]) #Make Predictions

  #Calculate Accuracy and F1 Metrics
  accuracy_dt = accuracy_score(test['Class'], predictions_dt) #Compare prediction to known label
  #print(f"Accuracy: {accuracy_dt}")
  f1_dt = f1_score(test['Class'], predictions_dt, pos_label = 'Malicious') #Compute F1 Score
  #print(f"F1 Score: {f1_dt}")

  #Make Confusion Matrix
  con_mtrx_dt = pd.crosstab(test['Class'], predictions_dt, rownames = ['Actual'], colnames = ['Predicted'])
  #display(con_mtrx_dt) #Display Confusion Matrix

  #True Positive, False Positive, False Negative, and True Negative
  # .loc[row (Actuals), column (Predictions)]
  true_pos_dt = con_mtrx_dt.loc['Malicious', 'Malicious']
  false_pos_dt = con_mtrx_dt.loc['Benign', 'Malicious']
  false_neg_dt = con_mtrx_dt.loc['Malicious', 'Benign']
  true_neg_dt = con_mtrx_dt.loc['Benign', 'Benign']

  #False Positive Rate
  false_pos_rate_dt = false_pos_dt / (false_pos_dt + true_neg_dt)
  #print(f"False Positive Rate: {false_pos_rate_dt}")

  #False Negative Rate
  false_neg_rate_dt = false_neg_dt / (false_neg_dt + true_pos_dt)
  #print(f"False Negative Rate: {false_neg_rate_dt}")

  #Store Results for Decision Tree Model
  results['decision_tree'] = {"Accuracy": accuracy_dt, "F1 Score": f1_dt, "False Positive Rate": false_pos_rate_dt, "False Negative Rate": false_neg_rate_dt, "Confusion Matrix": con_mtrx_dt}

  print("Decision Tree - Done.")


#print("\n--------------------------Logistic Regression------------------------------")
if 'train' not in locals() or 'test' not in locals():
  print("Error: 'train' and 'test' DataFrames are not defined. Please run the preceding cells to define them.")
else:
  model_lr = LogisticRegression(random_state = 0) #Specify the model used
  model_lr.fit(train[features], train['Class']) #Train the model
  predictions_lr = model_lr.predict(test[features]) #Make Predictions given X (features)

  #Calculate Accuracy and F1 Metrics
  accuracy_lr = accuracy_score(test['Class'], predictions_lr) #Compare prediction to known label
  #print(f"Accuracy: {accuracy_lr}")
  f1_lr = f1_score(test['Class'], predictions_lr, pos_label = 'Malicious') #Compute F1 Score
  #print(f"F1 Score: {f1_lr}")

  #Make Confusion Matrix
  con_mtrx_lr = pd.crosstab(test['Class'], predictions_lr, rownames = ['Actual'], colnames = ['Predicted'])
  #display(con_mtrx_lr) #Display Confusion Matrix

  # True Positive, False Positive, False Negative, and True Negative
  # .loc[row (Actuals), column (Predictions)]
  true_pos_lr = con_mtrx_lr.loc['Malicious', 'Malicious']
  false_pos_lr = con_mtrx_lr.loc['Benign', 'Malicious']
  false_neg_lr = con_mtrx_lr.loc['Malicious', 'Benign']
  true_neg_lr = con_mtrx_lr.loc['Benign', 'Benign']

  #False Positive Rate
  false_pos_rate_lr = false_pos_lr / (false_pos_lr + true_neg_lr)
  #print(f"False Positive Rate: {false_pos_rate_lr}")

  #False Negative Rate
  false_neg_rate_lr = false_neg_lr / (false_neg_lr + true_pos_lr)
  #print(f"False Negative Rate: {false_neg_rate_lr}")

  #Store Results for Logistic Regression in a Dictionary
  results['logistic_regression'] = {"Accuracy": accuracy_lr, "F1 Score": f1_lr, "False Positive Rate": false_pos_rate_lr, "False Negative Rate": false_neg_rate_lr, "Confusion Matrix": con_mtrx_lr}

  print("Logistic Regression - Done.")


#print("\n--------------------------Multi-Layer Perceptron------------------------------")
if 'train' not in locals() or 'test' not in locals():
  print("Error: 'train' and 'test' DataFrames are not defined. Please run the preceding cells to define them.")
else:
  #Note: MLPClassifer does not take n_jobs arguement
  model_mlp = MLPClassifier(random_state = 0) #Specify the model used
  model_mlp.fit(train[features], train['Class']) #Train the model
  predictions_mlp = model_mlp.predict(test[features]) #Make Predictions

  #Calculate Accuracy and F1 Metrics
  accuracy_mlp = accuracy_score(test['Class'], predictions_mlp) #Compare prediction to known label
  #print(f"Accuracy: {accuracy_mlp}")
  f1_mlp = f1_score(test['Class'], predictions_mlp, pos_label = 'Malicious') #Compute F1 Score
  #print(f"F1 Score: {f1_mlp}")

  #Make Confusion Matrix
  con_mtrx_mlp = pd.crosstab(test['Class'], predictions_mlp, rownames = ['Actual'], colnames = ['Predicted'])
  #display(con_mtrx_mlp) #Display Confusion Matrix

  # True Positive, False Positive, False Negative, and True Negative
  # .loc[row (Actuals), column (Predictions)]
  true_pos_mpl = con_mtrx_mlp.loc['Malicious', 'Malicious']
  false_pos_mpl = con_mtrx_mlp.loc['Benign', 'Malicious']
  false_neg_mpl = con_mtrx_mlp.loc['Malicious', 'Benign']
  true_neg_mpl = con_mtrx_mlp.loc['Benign', 'Benign']

  #False Positive Rate
  false_pos_rate_mpl = false_pos_mpl / (false_pos_mpl + true_neg_mpl)
  #print(f"False Positive Rate: {false_pos_rate_mpl}")

  #False Negative Rate
  false_neg_rate_mpl = false_neg_mpl / (false_neg_mpl + true_pos_mpl)
  #print(f"False Negative Rate: {false_neg_rate_mpl}")

  results['multi_layer_perceptron'] = {"Accuracy": accuracy_mlp, "F1 Score": f1_mlp, "False Positive Rate": false_pos_rate_mpl, "False Negative Rate": false_neg_rate_mpl, "Confusion Matrix": con_mtrx_mlp}

  print("Multi-Layer Perceptron - Done.")




Random Forest - Done.


Predicted,Benign,Malicious
Actual,,
Benign,184496,242
Malicious,262,74550


Decision Tree - Done.
Logistic Regression - Done.
Multi-Layer Perceptron - Done.


In [13]:
# Print the Outputs

for model in results.keys():
  print(f"--------- {model.title()} Results ----------")
  print(f"Accuracy: {results[model]['Accuracy']}")
  print(f"F1 Score: {results[model]['F1 Score']}")
  print(f"False Positive Rate: {results[model]['False Positive Rate']}")
  print(f"False Negative Rate: {results[model]['False Negative Rate']}")
  display(results[model]['Confusion Matrix'])
  print("\n")

--------- Random_Forest Results ----------
Accuracy: 0.9982700828356771
F1 Score: 0.9969990843531322
False Positive Rate: 0.001207114941159913
False Negative Rate: 0.0030209057370475323


Predicted,Benign,Malicious
Actual,,
Benign,184515,223
Malicious,226,74586




--------- Decision_Tree Results ----------
Accuracy: 0.9980581776151031
F1 Score: 0.9966311061201573
False Positive Rate: 0.0013099632993753315
False Negative Rate: 0.0035021119606480243


Predicted,Benign,Malicious
Actual,,
Benign,184496,242
Malicious,262,74550




--------- Logistic_Regression Results ----------
Accuracy: 0.8087767289539588
F1 Score: 0.635883440443701
False Positive Rate: 0.09829055202503004
False Negative Rate: 0.4207079078222745


Predicted,Benign,Malicious
Actual,,
Benign,166580,18158
Malicious,31474,43338




--------- Multi_Layer_Perceptron Results ----------
Accuracy: 0.7467231747254864
F1 Score: 0.21635990844936107
False Positive Rate: 5.413071485022031e-06
False Negative Rate: 0.8786959311340427


Predicted,Benign,Malicious
Actual,,
Benign,184737,1
Malicious,65737,9075


**Question #1: Write Up**

a): Which model has the highest classification accuracy?

**Answer**: Random Forest model had the highest classification accuracy at 99.827%. The Decision Tree model was a close second place with a classification accuracy of 99.805%.


---

b): Which model has the highest false positive rate?

**Answer**: The logistic regression model had the highest false positive rate at 9.82%.

---

c): Which model had the lowest false negative rate?

**Answer**: The random forest model had the lowest false negative rate at 0.302%.

---

d): If you have to make the descion to deploy one of these models to detect intrusions on a network, which would you select and why?

**Answer**: I would choose to deploy the Random Forest model for puposes of implementing an intrusion detection system. From the start, the Multi-Layer Perceptron and the Logistic Regression models have much lower accuracy and F1 scores compared to the other options, so I would eliminate the MLP and Logistic Regression models from consideration. This would leave a decision between the Random Forest and Decision Tree models. Both the Random Forest and Decision Tree models have similar accuracy and F1 scores, so I would look at the False Negative rate. In the scenario of implementing an intrusion detection system, a False Negative would mean incorrectly categorizing a message that is actually Malicious as Begnin. This could have severe consequences on the network, and therefore I would select the model with the lower false negative rate. In this case the Random Forest has a False-Negative rate of 0.302% versus the Decision Tree's False-Negative rate of 0.350%, so Random Forest would be the better model selection for this context.

# Question 2 (20pts.)
In any dataset, there is a subset of features that play a decisive role in a model's accuracy and others that play little/no role. [Feature selection methods](https://scikit-learn.org/stable/modules/feature_selection.html#feature-selection) perform automated feature importance analysis and produce a ranked list of features from which one can pick the top-$k$ features for training, where $k$ is empirically determined with respect to desired performance metrics (e.g., accuracy, false poitives, ...). Your task is to evaluate *Univariate feature selection* on the four models you compared in Question 1 and answer the following question:
 - Taking overall accuracy as a criterion, what is the subset of features that is the most effective? On which model?

**Note**: You need to write the relevant code and show execution outputs to justify your your answers.


In [14]:

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif


##########################  Part 1: Feature Ranking  ########################
grader = SelectKBest(score_func=f_classif, k='all')  #f_classif is ANOVA F-test from stats.
grader.fit(train[features], train['Class'])   #Calc F-Stat for each feature.

#Get Rankings
feature_scores = pd.DataFrame({'Feature': features, "Score": grader.scores_})
feature_scores = feature_scores.sort_values(by='Score', ascending=False)
# display(feature_scores)

#Most Important Features
print("Top 15 Most Important Features:")
display(feature_scores.head(15))
print("-------------------------------------")

print("\n")


Top 15 Most Important Features:


,Feature,Score
13,Bwd Packet Length Std,463791.754780
10,Bwd Packet Length Max,425317.056351
12,Bwd Packet Length Mean,419061.839614
54,Avg Bwd Segment Size,419061.839613
41,Packet Length Std,362298.779068
39,Max Packet Length,330590.212054
42,Packet Length Variance,324176.062622
40,Packet Length Mean,279015.185255
52,Average Packet Size,276638.817572
22,Fwd IAT Std,275223.077033


-------------------------------------




In [15]:
#Question 2 - Part 2: Test All Four Models With Different K Feature Subsets

models = {
    #Note: Decision Tree and MLPClassifier do not accept n_jobs arg.
    "Logistic Regression": LogisticRegression(random_state=0),
    "Decision Tree": DecisionTreeClassifier(random_state=0),
    "Random Forest": RandomForestClassifier(random_state=0),
    "Multi-Layer Perceptron": MLPClassifier(random_state=0)
}

# Try different numbers of top features
k_values = [5, 10, 15, 25, len(features)]

#Empty Dictionary to store results. Outer dictionary.
results_by_k_value = {}

for k in k_values:

  top_k_features = feature_scores.head(k)['Feature'].tolist()

  # Nested dictionary to hold data for each k number of features
  results_by_k_value[k] = {}

  for model_name, model in models.items():
    model.fit(train[top_k_features], train['Class'])  #Train the model
    predictions = model.predict(test[top_k_features]) #Make Predictions

    #Calculate Accuracy
    accuracy = accuracy_score(test['Class'], predictions) #Compare prediction to known label

    #Calculate F1
    f1 = f1_score(test['Class'], predictions, pos_label = 'Malicious') #Compute F1 Score

    #Make Confusion Matrix
    con_mtrx = pd.crosstab(test['Class'], predictions, rownames = ['Actual'], colnames = ['Predicted'])

    #True Positive, False Positive, False Negative, and True Negative
    # .loc[row (actual), column (predicted)]
    true_pos = con_mtrx.loc['Malicious', 'Malicious']
    false_pos = con_mtrx.loc['Benign', 'Malicious']
    false_neg = con_mtrx.loc['Malicious', 'Benign']
    true_neg = con_mtrx.loc['Benign', 'Benign']

    #False Positive Rate
    false_pos_rate = false_pos / (false_pos + true_neg)

    #False Negative Rate
    false_neg_rate = false_neg / (false_neg + true_pos)

    #Store Data For Each K and Model in Nested Dictionary
    results_by_k_value[k][model_name] = {
        "Accuracy": accuracy,
        "F1 Score": f1,
        "False Positive Rate": false_pos_rate,
        "False Negative Rate": false_neg_rate,
        "Confusion Matrix": con_mtrx
    }

    print(f"Training Complete for {model_name} with {k} Features.")

Training Complete for Logistic Regression with 5 Features.
Training Complete for Decision Tree with 5 Features.
Training Complete for Random Forest with 5 Features.
Training Complete for Multi-Layer Perceptron with 5 Features.
Training Complete for Logistic Regression with 10 Features.
Training Complete for Decision Tree with 10 Features.
Training Complete for Random Forest with 10 Features.
Training Complete for Multi-Layer Perceptron with 10 Features.
Training Complete for Logistic Regression with 15 Features.
Training Complete for Decision Tree with 15 Features.
Training Complete for Random Forest with 15 Features.
Training Complete for Multi-Layer Perceptron with 15 Features.
Training Complete for Logistic Regression with 25 Features.
Training Complete for Decision Tree with 25 Features.
Training Complete for Random Forest with 25 Features.
Training Complete for Multi-Layer Perceptron with 25 Features.
Training Complete for Logistic Regression with 78 Features.
Training Complete fo

In [16]:
#Print Question #2 Output For Each Model and K-Feature Combination

for k in results_by_k_value.keys():

  print(f"----------- Results for Top K={k} Features --------------")

  for model_name in results_by_k_value[k].keys():
    print(f"Model: {model_name} with {k} Features")
    print(f"Accuracy: {results_by_k_value[k][model_name]["Accuracy"]: }")
    #print(f"F1 Score: {results_by_k_value[k][model_name]["F1 Score"]: }")
    #print(f"False Positive Rate: {results_by_k_value[k][model_name]["False Positive Rate"]: }")
    #print(f"False Negative Rate: {results_by_k_value[k][model_name]["False Negative Rate"]: }")
    print("\n")

----------- Results for Top K=5 Features --------------
Model: Logistic Regression with 5 Features
Accuracy:  0.8147100751300328


Model: Decision Tree with 5 Features
Accuracy:  0.9378886534386438


Model: Random Forest with 5 Features
Accuracy:  0.937965709882489


Model: Multi-Layer Perceptron with 5 Features
Accuracy:  0.9315546137545753


----------- Results for Top K=10 Features --------------
Model: Logistic Regression with 10 Features
Accuracy:  0.8253746869581968


Model: Decision Tree with 10 Features
Accuracy:  0.9678327875168561


Model: Random Forest with 10 Features
Accuracy:  0.9679252552494703


Model: Multi-Layer Perceptron with 10 Features
Accuracy:  0.8127875168560971


----------- Results for Top K=15 Features --------------
Model: Logistic Regression with 15 Features
Accuracy:  0.7288769023309575


Model: Decision Tree with 15 Features
Accuracy:  0.9839876709689848


Model: Random Forest with 15 Features
Accuracy:  0.9842535157002504


Model: Multi-Layer Perceptron

**Question #2 Write Up**

Prompt: Taking overall accuracy as a criterion, what is the subset of features that is the most effective? On which model?

**Answer:**
The Random Forest model with all (k=78) features resulted in the highest accuracy rating of 99.83% out of all the model and k-feature combinations tested. One interesting observation was that each model responded differently to the increasing number of k-features. The Multi-Layer Perceptron model started with a reasonably high accuracy of 93% when k=5, then dropped to 81-82% accuracy with k-features between 10 and 25, and then dropped further to only 71.28% accuracy when all k-features were included. The Logistic Regression model started with an accuracy score of ~81% when k=5, then dropped to ~72% when k=15, increased to ~75% when k=25, and then returned to approximatley 81% accuracy when all k=78 features were included. Lastly, both the Decision Tree and Random Forest models appeared to improve in accuracy as the number of k-features increased.

# Question 3 (10pts.)
It is often recommended that we [*normalize*](https://scikit-learn.org/stable/modules/preprocessing.html#normalization) feature values to avoid sparseness of a dataset (hence skewness of a model). Repeat what you did for Question 1 with noramalized features and determine whether you get different answers to questions a) to d).

**Note**: You need to write the relevant code and show execution outputs to justify your your answers.

In [17]:
#Note: Google reccomended using StandardScaler instead of MinMaxScaler,
# as MinMaxScaler can compress data if there is an outlier.
from sklearn.preprocessing import StandardScaler


# Step #1: Scale the Feature Values
# Create the Scaler Object
scaler = StandardScaler()

# Make copies of the original data frames
train_normalized = train.copy()
test_normalized = test.copy()

# Train the scaler / normalizer object on the features. Fit the scaler to training set only.
# NOTE: Per Google, scaling the entire dataset can lead to "data leakage".
# Only fit the scaler to the training data set.
scaler.fit(train[features])

# Then apply the scaler transformation to both the training and test data sets.
# Note: The scaled "features" column is overwritten in normalized dataframes. Othere data remains the same (from copying).
train_normalized[features] = scaler.transform(train[features])
test_normalized[features] = scaler.transform(test[features])


#Dictionary to Store Results
results_normalized = {}

models = {
    "Random Forest": RandomForestClassifier(random_state=0),
    "Decision Tree": DecisionTreeClassifier(random_state=0),
    "Logistic Regression": LogisticRegression(random_state=0),
    "Multi-Layer Perceptron": MLPClassifier(random_state=0)
}

for model_name, model in models.items():

  # print(f"-----------Normalized {model_name} Results---------------------")

  model.fit(train_normalized[features], train_normalized['Class'])  #Train the model
  predictions_normalized = model.predict(test_normalized[features]) #Make Predictions

  # Calculate Metrics
  accuracy_normalized = accuracy_score(test_normalized['Class'], predictions_normalized) #Compare prediction to known "Class" label
  # print(f"Accuracy (normalized): {accuracy_normalized}")

  f1_normalized = f1_score(test_normalized['Class'], predictions_normalized, pos_label = 'Malicious') #Compute F1 Score
  # print(f"F1 Score (normalized): {f1_normalized}")

  #Confusion Matrix
  con_mtrx_normalized = pd.crosstab(test_normalized['Class'], predictions_normalized, rownames = ['Actual'], colnames = ['Predicted'])
  # display(con_mtrx_normalized) #Display Confusion Matrix

  true_pos_normalized = con_mtrx_normalized.loc['Malicious', 'Malicious']
  false_pos_normalized = con_mtrx_normalized.loc['Benign', 'Malicious']
  false_neg_normalized = con_mtrx_normalized.loc['Malicious', 'Benign']
  true_neg_normalized = con_mtrx_normalized.loc['Benign', 'Benign']

  false_pos_rate_normalized = false_pos_normalized / (false_pos_normalized + true_neg_normalized)
  # print(f"False Positive Rate (normalized): {false_pos_rate_normalized}")

  false_neg_rate_normalized = false_neg_normalized / (false_neg_normalized + true_pos_normalized)
  # print(f"False Negative Rate (normalized): {false_neg_rate_normalized}")

  results_normalized[model_name] = {
      "Accuracy": accuracy_normalized,
      "F1 Score": f1_normalized,
      "False Positive Rate": false_pos_rate_normalized,
      "False Negative Rate": false_neg_rate_normalized,
      "Confusion Matrix": con_mtrx_normalized

  }

  print(f"Done normalizing {model_name}.")


Done normalizing RandomForestClassifier(random_state=0).
Done normalizing DecisionTreeClassifier(random_state=0).
Done normalizing LogisticRegression(random_state=0).
Done normalizing MLPClassifier(random_state=0).


In [18]:
print("==================== Normalized Results ====================")

for model in results_normalized.keys():
  print(f"----------- Normalized {model} ---------------------")
  print(f"Model: {model}")
  print(f"Accuracy (normalized): {results_normalized[model]['Accuracy']}")
  print(f"F1 Score (normalized): {results_normalized[model]['F1 Score']}")
  print(f"False Positive Rate (normalized): {results_normalized[model]['False Positive Rate']}")
  print(f"False Negative Rate (normalized): {results_normalized[model]['False Negative Rate']}")
  print("\n")


print("================ Original Model Results (From Question #1)===================")

for model in results.keys():
  print(f"--------- Original {model} (Non-Normalized) Recap ----------")
  print(f"Model: {model}")
  print(f"Accuracy (original): {results[model]['Accuracy']}")
  print(f"F1 Score (original): {results[model]['F1 Score']}")
  print(f"False Positive Rate (original): {results[model]['False Positive Rate']}")
  print(f"False Negative Rate (original): {results[model]['False Negative Rate']}")
  print("\n")

==================== Normalized Results ====================
----------- Normalized Random Forest ---------------------
Model: Random Forest
Accuracy (normalized): 0.9982662300134849
F1 Score (normalized): 0.9969923002887392
False Positive Rate (normalized): 0.0011962887981898688
False Negative Rate (normalized): 0.003061006255680907


----------- Normalized Decision Tree ---------------------
Model: Decision Tree
Accuracy (normalized): 0.9980620304372954
F1 Score (normalized): 0.9966376780593453
False Positive Rate (normalized): 0.0012937240849202655
False Negative Rate (normalized): 0.0035288456397369406


----------- Normalized Logistic Regression ---------------------
Model: Logistic Regression
Accuracy (normalized): 0.9207513003274899
F1 Score (normalized): 0.864369786686888
False Positive Rate (normalized): 0.06116770778074895
False Negative Rate (normalized): 0.12389723573758221


----------- Normalized Multi-Layer Perceptron ---------------------
Model: Multi-Layer Perceptron
A

**Question #3 Write Up:**

Prompt: It is often reccomended that we normalize feature values to avoid sparseness of a dataset (and skewness of a model). Repeat what you did for Question #1 with normalized features and determine whether you get different answers to questions a) through d).

---

a) Which model has the highest classification accuracy (after normalization)?

The Random Forest model remained the most accurate model after normalization of the dataset with any accuracy score of 99.826%.

---

b) Which (normalized) model has the highest False Positive rate?

The Logistic Regression model remained the model with the highest False Positive rate after normalization.

---

c) Which model has the lowest False Negative rate?

After normalizing the dataset, the Random Forest model remained as the model with lowest False Negative rate of 0.306%.

---

d) If you had to make the decision of which model to deploy to detect intrusions on a network, which would you pick and why?

From the normalized models, I would still select the Random Forest model. After normalizing the dataset, the Random Forest, Decision Tree, and Multi-Layer Perceptron models all scored above 99% in accuracy. Then comparing the False Negative rates, the MLP has a FNR = 0.93%, the Decision Tree model has an FNR = 0.35%, and the Random Forest has an FNR = 0.306%. Because the normalized Random Forest model has a high degree of accuracy (comparable to the others) and the lowest False Negative rate, it would be the best model to choose in the context of an intrusion dection system classifer, because False Negatives are potentially harmful to the network.

---

**Additional Observations:**

After normalizing the dataset, the accuracy scores of both the Logistic Regression and the Multi-Layer Perceptron models improved significantly. Before normalization, the Logistic Regression model scored 80.8% accuracy and the MLP model scored only 74.6% accuracy. After normalizing the data set, the Logistic Regression model's accuracy improved to 92% and the MLP model's accuracy improved to 99.4%. The False Negative rates for both models also decreased signficantly after normalization, and both model's had increased F1 Score's after normalization. Overall normalizing the dataset appears to have improved the performance of the Logistic Regression and Multi-Layer Perceptron models, while having minimal impact on the Random Forest and Decsion Tree model results.



